In [ ]:
import pandas as pd

# 1. Import des données

In [ ]:
df_irve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435'
)

print(f"Shape : {df_irve.shape}")
df_irve.head()

Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).
Cette table comporte 210129 lignes pour 52 variables.

In [ ]:
df_ve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
 encoding='latin-1'
)

print(f"Shape : {df_ve.shape}")
df_ve.head()

Ce second jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge. Il contient l'historique des immatriculations pour chaque commune à différentes dates.
Cette table comporte 703545 lignes pour 8 variables. 

In [ ]:
df_revenus = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',sep=';'
)

print(f"Shape : {df_revenus.shape}")
df_revenus.head()

Ce dernier jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.
Cette table comporte 34926 lignes pour 57 variables.

# 2. Jointure des tables

## 2.1 Choix de la variable de jointure

Les 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus

La jointure sera faite sur ces variables.

## 2.2 Préparation de la variable de jointure

Le diagnostic met en évidence plusieurs points d’attention :
- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

### 2.2.1 Unification des codes

In [ ]:
from src.preparation_data import nettoyer_code_insee

In [ ]:
df_irve["code_insee_commune"] = df_irve["code_insee_commune"].apply(nettoyer_code_insee)
df_ve["CODGEO"] = df_ve["CODGEO"].apply(nettoyer_code_insee)
df_revenus["Code géographique"] = df_revenus["Code géographique"].apply(nettoyer_code_insee)

Les codes insee sont maintenant au même format dans les 3 bases de données : un code à 5 caractères.

### 2.2.2 Compléter les valeurs manquantes

In [ ]:
from src.preparation_data import creer_gdf_irve, charger_communes, joindre_communes, ajouter_codes_geo

In [ ]:
gdf_irve = creer_gdf_irve(df_irve, "consolidated_longitude", "consolidated_latitude")
gdf_communes = charger_communes()
gdf_result = joindre_communes(gdf_irve, gdf_communes)
df_irve = ajouter_codes_geo(df_irve, gdf_result)

Nous avons maintenant 2 colonnes concernant le code géographique dans la base de données IRVE :
- `code_insee_commune` : la variable initiale
- `code_geo_total` : la variable recalculant tous les codes géographiques à partir de la latitude et la longitude

In [ ]:
from src.preparation_data import compter_valeurs_manquantes, compter_uniques

In [ ]:
colonnes_irve = ["code_insee_commune", "code_geo_total"]

print("Valeurs manquantes :")
print(compter_valeurs_manquantes(df_irve, colonnes_irve))

print("\nValeurs uniques :")
print(compter_uniques(df_irve, colonnes_irve))

Cette manipulation permet de passer de 57919 valeurs manquantes à 1768. De plus les codes recalculés sont plus fiables que ceux initiaux.

## 2.3 Jointure

### 2.3.1 Une ligne par code géo

Il faut réfléchir à la façon dont on veut agréger df_irve et quelles variables on souhaite garder.

##### df_irve

In [ ]:
# Nombre de points de recharge par commune
stats_irve_com = df_irve.groupby('code_geo_total').size().reset_index(name='nb_points_recharge')

# puissance totale disponible par commune
stats_puissance_com = df_irve.groupby('code_geo_total')['puissance_nominale'].sum().reset_index(name='puissance_totale_kw')

# Fusion des deux métriques IRVE
df_irve_final = pd.merge(stats_irve_com, stats_puissance_com, on='code_geo_total')
df_irve_final.head()

##### df_ve

Pour que l'analyse soit cohérente avec les données de bornes (IRVE) et de l'INSEE (qui sont des "clichés" à un instant T), il faut isoler la situation la plus récente pour chaque commune afin d'avoir une unique ligne par code insee.

In [ ]:
df_ve["DATE_ARRETE"].dtype

In [ ]:
ve = df_ve.copy()

# Conversion en format date pour être sûr du tri
ve['DATE_ARRETE'] = pd.to_datetime(ve['DATE_ARRETE'])

# Tri par commune puis par date (de la plus ancienne à la plus récente)
ve = ve.sort_values(['CODGEO', 'DATE_ARRETE'])

# garde que la DERNIÈRE ligne pour chaque CODGEO (la plus récente)
df_ve_latest = ve.drop_duplicates(subset='CODGEO', keep='last')

df_ve_latest

##### df_revenus

In [ ]:
# On garde le code commune et le revenu médian (souvent colonne 'MED21' pour 2021)
df_revenus_final = df_revenus[['COM', 'MED21']].copy()
df_revenus_final.rename(columns={'COM': 'code_commune_insee', 'MED21': 'revenu_median'}, inplace=True)